# Bluesky Bot-Cluster Analysis

This notebook produces a complete bot-cluster report around a Bluesky anchor account. The only mandatory configuration is `ANCHOR_HANDLE` in the parameters cell below.

**Prerequisites**
- Notebook is bound to a Bluesky KQL database (see KQL tab on the left). `deploy-fabric-notebook.ps1` wires this up automatically.
- The identity running the notebook has Reader rights on the Eventhouse database.


In [ ]:
# Install module from the real-time-sources repo (SHA-pinned)
%pip install -q --upgrade --force-reinstall --no-cache-dir --no-deps git+https://github.com/clemensv/real-time-sources@8aacac0bc9fd215b8cf556731333afec0cf61291#subdirectory=bluesky/botfinder
%pip install -q git+https://github.com/clemensv/real-time-sources@8aacac0bc9fd215b8cf556731333afec0cf61291#subdirectory=bluesky/botfinder


In [ ]:
# === PARAMETERS ===
# Main knob: ANCHOR_HANDLE.
# KUSTO_URI / KUSTO_DATABASE are populated by the deploy script per workspace.
ANCHOR_HANDLE  = "niusde.bsky.social"
LOOKBACK_DAYS  = 90
MAX_FOLLOWERS  = 5000
ENRICH_LIMIT   = 400
SKIP_API       = False

# Workspace-specific defaults - overwritten by the deploy script.
KUSTO_URI      = ""
KUSTO_DATABASE = ""


In [ ]:
from botfinder import Config, run_full_pipeline

config = Config.from_fabric_context(
    anchor_handle=ANCHOR_HANDLE,
    kusto_uri=KUSTO_URI or None,
    kusto_database=KUSTO_DATABASE or None,
    lookback_days=LOOKBACK_DAYS,
    max_followers=MAX_FOLLOWERS,
    enrich_limit=ENRICH_LIMIT,
    skip_api=SKIP_API,
)
print(f"Cluster:  {config.kusto_uri}")
print(f"Database: {config.kusto_database}")
print(f"Anchor:   @{config.anchor_handle}")
assert config.kusto_uri and config.kusto_database, (
    "KQL database not configured. Re-deploy this notebook via "
    "deploy-fabric-notebook.ps1 or set KUSTO_URI / KUSTO_DATABASE "
    "in the parameters cell."
)


## Run the full pipeline
Acquire (KQL) -> API enrichment -> scoring -> cluster graph -> cross-follow -> cards.

In [ ]:
result = run_full_pipeline(config)
scores = result.analysis.scores_df
print(f"Cohort: {len(scores)} accounts")
print(f"Bots:   {(scores['score'] >= config.bot_score_threshold).sum()}")


## KPI overview

In [ ]:
import pandas as pd
scores = result.analysis.scores_df
flags = scores['flags'].fillna('')
kpis = pd.DataFrame([
    {'Metric': 'Cohort total',        'Value': len(scores)},
    {'Metric': 'Bots (score>=0.5)',   'Value': int((scores['score'] >= 0.5).sum())},
    {'Metric': 'Instant follow',      'Value': int(flags.str.contains('INSTANT_FOLLOW').sum())},
    {'Metric': 'Anonymous profiles',  'Value': int(flags.str.contains('ANONYMOUS_PROFILE').sum())},
    {'Metric': 'Deleted profiles',    'Value': int(flags.str.contains('DELETED').sum())},
    {'Metric': 'Cluster nodes',       'Value': len(result.cluster_nodes)},
    {'Metric': 'Cluster edges',       'Value': len(result.cluster_edges)},
])
display(kpis)


## Top suspects

In [ ]:
top = scores.nlargest(20, 'score')[['handle', 'display_name', 'score', 'flags']]
display(top)


## Cards
Each card is a standalone Plotly visualization in 1200x675 format (card 5 is 1200x1200).

In [ ]:
for card_id, fig in result.cards.items():
    print(card_id)
    fig.show()


## Write HTML dossier to the lakehouse (optional)

In [ ]:
from pathlib import Path
from botfinder.dossier import render_dossier

try:
    out = Path('/lakehouse/default/Files/botfinder')
    out.mkdir(parents=True, exist_ok=True)
    figures = list(result.cards.values())
    flags = scores['flags'].fillna('')
    total = len(scores)
    stats = {
        'total_suspects':         total,
        'high_confidence_bots':   int((scores['score'] >= 0.7).sum()),
        'medium_confidence_bots': int(((scores['score'] >= 0.4) & (scores['score'] < 0.7)).sum()),
        'pct_instant_follow':     100 * flags.str.contains('INSTANT_FOLLOW').sum() / max(total, 1),
        'pct_anonymous':          100 * flags.str.contains('ANONYMOUS_PROFILE').sum() / max(total, 1),
        'mean_score':             float(scores['score'].mean()),
        'p90_score':              float(scores['score'].quantile(0.9)),
    }
    suspects = []
    for _, row in scores.nlargest(50, 'score').iterrows():
        suspects.append({
            'handle':          row.get('handle') or '',
            'display_name':    row.get('display_name') or '',
            'total_score':     float(row['score']),
            'flags':           [f for f in (row.get('flags') or '').split(',') if f],
            'posts_count':     0, 'followers_count': 0, 'age_minutes': -1,
            'source':          row.get('source') or '',
        })
    path = out / f'dossier_{config.anchor_slug}.html'
    render_dossier(config.anchor_handle, stats, figures, suspects, config.lookback_days, path)
    print(f'Dossier written to {path}')
except Exception as e:
    print(f'(Lakehouse write skipped: {e})')
